In [1]:
# Load environment variables and verify the project setup.
import sys
from pathlib import Path

# Find the repo root (the folder containing env_checker.py) and make it importable.
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "env_checker.py").exists())
sys.path.insert(0, str(ROOT))

# Load .env into the environment for this session.
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except ModuleNotFoundError:
    print("python-dotenv not installed yet — run: uv add python-dotenv")

# Verify .env variables and required packages.
from env_checker import run_checks
run_checks()

Environment variables (from .env.example)
  ✓ OPENAI_API_KEY  — set
  ✓ ANTHROPIC_API_KEY  — set
  ✓ LANGSMITH_TRACING  — set
  ✓ LANGSMITH_ENDPOINT  — set
  ✓ LANGSMITH_API_KEY  — set
  ✓ LANGSMITH_PROJECT  — set
  ✓ CHROMA_PERSIST_DIR  — set

Required packages (from pyproject.toml)
  ✓ beautifulsoup4  — installed (4.14.3)
  ✓ chromadb  — installed (1.5.9)
  ✓ langchain  — installed (1.3.2)
  ✓ langchain-chroma  — installed (1.1.0)
  ✓ langchain-classic  — installed (1.0.7)
  ✓ langchain-community  — installed (0.4.2)
  ✓ langchain-core  — installed (1.4.0)
  ✓ langchain-experimental  — installed (0.4.2)
  ✓ langchain-openai  — installed (1.2.2)
  ✓ lxml  — installed (6.1.1)
  ✓ onnxruntime  — installed (1.19.2)
  ✓ pypdf  — installed (6.12.2)
  ✓ python-dotenv  — installed (1.2.2)
  ✓ ragas  — installed (0.4.3)
  ✓ rank-bm25  — installed (0.2.2)
  ✓ rapidfuzz  — installed (3.14.5)
  ✓ ipykernel  — installed (7.2.0)
  ✓ jupyterlab  — installed (4.5.7)

✓ All checks passed.


True

# Naive RAG with BM25 (sparse retrieval)

This is the BM25 counterpart to the dense Naive RAG notebook. Same pipeline, **same three sections** — the only change is the retriever: **sparse/lexical BM25** instead of dense vector search over Chroma.

BM25 ranks chunks by exact term-frequency statistics (TF-IDF with length normalization). It uses **no embeddings or vector store at retrieval time** — fast, free, and strong on keywords, IDs, and rare terms, where dense search can be weak.

Sections:

1. **Ingestion** — load the PDF, semantically chunk it, and build an in-memory **BM25 index** (no Chroma).
2. **Inference** — retrieve relevant chunks with BM25 and generate an answer with the same LCEL chain (`retriever | prompt | llm | parser`).
3. **Evaluation** — score the pipeline with faithfulness & context precision — identical to the dense notebook, so the numbers are directly comparable.

## Section 1 — Ingestion

Build the knowledge base once: **load → chunk → index**. Unlike the dense notebook, there is no embedding/Chroma write step — BM25 indexes the chunk text directly.

### 1. Load the PDF

`PyPDFLoader` (backed by `pypdf`) returns one `Document` per page, each with `.page_content` and `.metadata` (source, page number).

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = ROOT / "assets/sample-docs/sdlc-end-to-end.pdf"

loader = PyPDFLoader(str(pdf_path))
documents = loader.load()
print(f"Loaded {len(documents)} page(s)")

/tmp/claude-501/ipykernel_79660/2466840251.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 12 page(s)


### 2. Semantic chunking

`SemanticChunker` embeds each sentence and breaks where the similarity between consecutive sentences drops, so each chunk holds one coherent idea and sizes vary with the content. It calls the embedding model while splitting (extra cost/latency) — we reuse the same `OpenAIEmbeddings` model for chunking and storage.

In [3]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

# One embedding model, reused for semantic chunking (below) and the vector store.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

splitter = SemanticChunker(
    embedding_model,
    breakpoint_threshold_type="percentile",
)

# Embeds sentences and splits at semantic boundaries (makes embedding API calls).
chunks = splitter.split_documents(documents)

sizes = [len(c.page_content) for c in chunks]
print(f"Split {len(documents)} pages into {len(chunks)} semantic chunks")
print(f"chunk size (chars) -> min {min(sizes)}, max {max(sizes)}, avg {sum(sizes)//len(sizes)}")

/tmp/claude-501/ipykernel_79660/2545116733.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Split 12 pages into 23 semantic chunks
chunk size (chars) -> min 150, max 1968, avg 802


### 3. Build the BM25 index (sparse)

The dense notebook embeds the chunks and writes them to Chroma here. BM25 instead builds an **in-memory lexical index** straight over the chunk text — no embeddings, no vector store, no persistence.

> Note: semantic chunking in step 2 still used OpenAI embeddings *to decide split points*. The BM25 retriever itself stays purely lexical.

In [4]:
from langchain_community.retrievers import BM25Retriever

# In-memory BM25 index over the chunks (replaces the Chroma vector store).
bm25_retriever = BM25Retriever.from_documents(chunks)
print(f"Built BM25 index over {len(chunks)} chunks")

Built BM25 index over 23 chunks


## Section 2 — Inference (Retrieval + LLM)

Answer a question by wiring **retriever → prompt → llm → parser** into a single LCEL chain. The retriever pulls the top-k chunks from Chroma; the prompt grounds the LLM on that context.

### 1. Retriever

BM25 exposes the same `.invoke(query) -> [Document]` interface as the dense retriever, so the rest of the chain is unchanged. We just set how many chunks to return.

In [5]:
bm25_retriever.k = 3      # number of chunks to return
retriever = bm25_retriever

### 2. Prompt template

Ground the model on the retrieved context and tell it to say so when the answer isn't there — the core of faithful RAG.

In [6]:
from langchain_core.prompts import ChatPromptTemplate

prompt = """
Use only the context provided to answer the following question.
If the answer is not in the context, reply that you are unsure.

Context: {context}

Question: {question}
"""

prompt_template = ChatPromptTemplate.from_template(prompt)

### 3. LLM

In [7]:
from langchain.chat_models import init_chat_model

llm = init_chat_model(model="gpt-5-nano")

### 4. LCEL chain

`question` flows through unchanged via `RunnablePassthrough`, while `retriever` fetches the context. Both fill the prompt, which feeds the LLM, and `StrOutputParser` unwraps the message to a plain string.

In [8]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

### 5. Run it

In [9]:
question = "What is the Low-Level Design (LLD)?"
response = chain.invoke(question)
print(response)

Low-Level Design (LLD) provides the detailed internal design of each component—the blueprint developers code from. It includes:
- Class/module structure, methods and responsibilities
- Detailed algorithms, data structures and pseudo-code
- Database tables, fields and relationships at field level
- API endpoint signatures, request/response schemas
- Sequence of operations, error handling and config details

Owner: Senior Developers / Tech Lead; Input: HLD; Flows to: Implementation.


## Section 3 — Evaluation (Faithfulness & Context Precision)

Two complementary RAGAS metrics score the pipeline we just built — reusing the `question`, `response`, and `retriever` from Section 2:

- **Faithfulness** — of the claims in the *answer*, what fraction are supported by the *retrieved context*? Low = hallucination.
- **Context precision** — were the *relevant* retrieved chunks ranked **near the top**? Low = the retriever buried the useful context among noise.

Both are **LLM-as-judge** metrics, so we point RAGAS at a judge model (`gpt-4o-mini`).

### Setup — RAGAS shim + evaluation inputs

> `ragas` 0.4.x imports `langchain_community.chat_models.vertexai.ChatVertexAI`, a submodule removed in `langchain-community` 0.4.x. ragas only imports the *name* (never instantiates it unless you use Vertex AI), so we register a tiny stub to satisfy the import. **This must run before importing `ragas`.**

In [10]:
import sys, types

# RAGAS compatibility shim — must run before importing ragas.
if "langchain_community.chat_models.vertexai" not in sys.modules:
    _shim = types.ModuleType("langchain_community.chat_models.vertexai")
    _shim.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _shim

from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

# Judge model RAGAS uses to score the metrics.
eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))

# The exact contexts the retriever returned for our question (what the answer is graded against).
retrieved = retriever.invoke(question)
retrieved_contexts = [d.page_content for d in retrieved]
print(f"Evaluating answer over {len(retrieved_contexts)} retrieved context(s)")

/Users/Prabhukumar/Projects/PycharmProjects/rag-reference/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Evaluating answer over 3 retrieved context(s)


/tmp/claude-501/ipykernel_79660/2992009439.py:14: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  eval_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))


### 1. Faithfulness

`Faithfulness` splits `response` into atomic claims and checks each against `retrieved_contexts`. Score = supported claims / total claims (1.0 = every claim is grounded).

In [11]:
from ragas.metrics import Faithfulness

faithfulness = Faithfulness(llm=eval_llm)

sample = SingleTurnSample(
    user_input=question,
    response=response,
    retrieved_contexts=retrieved_contexts,
)

# Top-level await works in the Jupyter kernel.
faithfulness_score = await faithfulness.single_turn_ascore(sample)
print(f"Faithfulness = {faithfulness_score:.3f}")

/tmp/claude-501/ipykernel_79660/1380331223.py:1: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness


Faithfulness = 1.000


### 2. Context precision

`LLMContextPrecisionWithoutReference` uses the generated `response` (no hand-written reference needed): for each retrieved context it judges whether that context helped produce the answer, then computes rank-weighted precision. Near 1.0 means the relevant chunks are ranked first.

In [12]:
from ragas.metrics import LLMContextPrecisionWithoutReference

context_precision = LLMContextPrecisionWithoutReference(llm=eval_llm)

sample = SingleTurnSample(
    user_input=question,
    response=response,
    retrieved_contexts=retrieved_contexts,
)

context_precision_score = await context_precision.single_turn_ascore(sample)
print(f"Context precision = {context_precision_score:.3f}")

/tmp/claude-501/ipykernel_79660/1029481067.py:1: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import LLMContextPrecisionWithoutReference


Context precision = 0.500


### Summary

In [13]:
print(f"Question          : {question}")
print(f"Faithfulness      : {faithfulness_score:.3f}  (answer claims supported by context)")
print(f"Context precision : {context_precision_score:.3f}  (relevant chunks ranked near the top)")

Question          : What is the Low-Level Design (LLD)?
Faithfulness      : 1.000  (answer claims supported by context)
Context precision : 0.500  (relevant chunks ranked near the top)
